# 01 — Diagnóstico Temporal e Delineamento Amostral

## TL;DR

A base apresenta mudanças temporais materiais: a taxa do evento sobe de 10,85% em 2019-01 para 17,47% em 2020-01; `cat_var13` reduz missing de 78,33% para 23,21%; e variáveis categóricas apresentam drift exploratório. Este notebook compara os cenários 8/2/3 e 9/2/2 sem utilizar modelos. A recomendação técnica permanece **pendente de validação humana**. Antes da modelagem, é indispensável validar a disponibilidade temporal das features e a origem dos códigos especiais de `var12`.

## Contexto e métodos

**Pergunta central:** A população e as variáveis apresentam estabilidade suficiente ao longo das safras para construirmos um modelo que generalize no tempo?

### Premissas principais

- `Ever30Mob6 = 1` significa somente ocorrência do evento adverso observado.
- `data_ref_safra` é o eixo temporal e `id` é a unidade observada.
- O PSI é diagnóstico: mede mudança de distribuição, não qualidade preditiva nem causalidade. Seus valores não serão tratados como cortes absolutos.
- Para evitar privilegiar um dos splits candidatos, 2019-01 a 2019-08 — janela comum de treino — é a referência do PSI; 2019-09 a 2020-01 é comparado mês a mês.
- Missing e os códigos `99997`, `99998` e `99999` de `var12` são bins separados no PSI. Eles não são transformados nem interpretados semanticamente.
- A disponibilidade das features no momento do score não pode ser confirmada apenas por nomes anonimizados.
- Nesta etapa todas as safras são vistas apenas para compreender população, drift e desenhar a validação. Após aprovação do split, o OOT será congelado e não poderá orientar feature selection, tratamento guiado pelo target, tuning, hiperparâmetros ou escolha do champion.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src' / 'behavior_score').exists():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src' / 'behavior_score').exists():
    raise RuntimeError('Execute a partir da raiz do projeto ou da pasta notebooks/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.behavior_score.config import (
    ALVO, COLUNA_ID, COLUNA_SAFRA, PASTA_DADOS_BRUTOS, PASTA_FIGURAS,
    PASTA_TABELAS, VARIAVEIS_CATEGORICAS, VARIAVEIS_MODELO, VARIAVEIS_NUMERICAS,
)
from src.behavior_score.visualization import (
    CORES, aplicar_eixo_percentual, aplicar_layout_executivo, salvar_grafico,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
arquivos_excel = sorted(PASTA_DADOS_BRUTOS.glob('*.xlsx'))
if len(arquivos_excel) != 1:
    raise RuntimeError(f'Esperado exatamente um Excel em data/raw; encontrados: {len(arquivos_excel)}')
caminho_base = arquivos_excel[0]
arquivo_excel = pd.ExcelFile(caminho_base, engine='openpyxl')
if arquivo_excel.sheet_names != ['case']:
    raise RuntimeError(f'Esperada a aba case; encontradas: {arquivo_excel.sheet_names}')
base = pd.read_excel(caminho_base, sheet_name='case', index_col=None, engine='openpyxl')
base_original = base.copy(deep=True)
base['safra'] = pd.to_datetime(base[COLUNA_SAFRA].astype('Int64').astype('string'), format='%Y%m', errors='coerce')
if base['safra'].isna().any():
    raise ValueError('Há safras inválidas no formato AAAAMM.')
safras = sorted(base['safra'].unique())
print(f'Fonte: {caminho_base.relative_to(RAIZ)} | aba: case | index_col: None | {len(base):,} registros | {len(safras)} safras')

Fonte: data\raw\base_behavior_score.xlsx | aba: case | index_col: None | 200,043 registros | 13 safras


## Dados

### A cronologia, a granularidade e o alvo foram preservados?

**Problema:** um split temporal só é defensável se as safras forem válidas e completas, a unidade não se repetir entre períodos e o alvo tiver denominador consistente.

In [2]:
tabela_safras = (base.groupby('safra', as_index=False)[ALVO]
                  .agg(quantidade_registros='size', quantidade_eventos='sum', taxa_evento='mean'))
sequencia_esperada = pd.date_range(tabela_safras['safra'].min(), tabela_safras['safra'].max(), freq='MS')
lacunas = sequencia_esperada.difference(pd.DatetimeIndex(tabela_safras['safra']))
validacoes = pd.DataFrame({
    'validacao': ['coluna_temporal', 'intervalo', 'safras', 'lacunas_mensais', 'ids_unicos',
                  'ids_repetidos', 'um_registro_por_id', 'target_binario_sem_missing',
                  'base_original_ordenada'],
    'resultado': [COLUNA_SAFRA, f"{tabela_safras.safra.min():%Y-%m} a {tabela_safras.safra.max():%Y-%m}",
                  len(tabela_safras), len(lacunas), base[COLUNA_ID].nunique(),
                  base[COLUNA_ID].duplicated(keep=False).sum(), base[COLUNA_ID].is_unique,
                  set(base[ALVO].unique()) == {0, 1} and not base[ALVO].isna().any(),
                  pd.to_datetime(base_original[COLUNA_SAFRA].astype(str), format='%Y%m').is_monotonic_increasing],
})
display(validacoes)
display(tabela_safras.style.format({'safra': lambda x: x.strftime('%Y-%m'), 'taxa_evento': '{:.2%}'}))
assert len(base) == tabela_safras['quantidade_registros'].sum()
assert int(base[ALVO].sum()) == int(tabela_safras['quantidade_eventos'].sum())
display(Markdown(
    f"**Análise/Interpretação:** As **{len(tabela_safras)} safras** são mensais e não têm lacunas. "
    f"Os **{base[COLUNA_ID].nunique():,} IDs** são únicos, logo a granularidade observada é um registro por ID, "
    "sem clientes repetidos entre safras. A ordem física original é embaralhada, mas a análise usa ordenação cronológica explícita."
))

,validacao,resultado
0,coluna_temporal,data_ref_safra
1,intervalo,2019-01 a 2020-01
2,safras,13
3,lacunas_mensais,0
4,ids_unicos,200043
5,ids_repetidos,0
6,um_registro_por_id,True
7,target_binario_sem_missing,True
8,base_original_ordenada,False


,safra,quantidade_registros,quantidade_eventos,taxa_evento
0,2019-01,13936,1512,10.85%
1,2019-02,13838,1601,11.57%
2,2019-03,13783,1598,11.59%
3,2019-04,13825,1675,12.12%
4,2019-05,14182,1662,11.72%
5,2019-06,14503,1734,11.96%
6,2019-07,14808,1737,11.73%
7,2019-08,15449,1819,11.77%
8,2019-09,16134,1993,12.35%
9,2019-10,16487,2100,12.74%


**Análise/Interpretação:** As **13 safras** são mensais e não têm lacunas. Os **200,043 IDs** são únicos, logo a granularidade observada é um registro por ID, sem clientes repetidos entre safras. A ordem física original é embaralhada, mas a análise usa ordenação cronológica explícita.

In [3]:
fig_volume = go.Figure(go.Bar(
    x=tabela_safras['safra'], y=tabela_safras['quantidade_registros'], marker_color=CORES['principal'],
    hovertemplate='Safra: %{x|%Y-%m}<br>Registros: %{y:,}<extra></extra>',
))
aplicar_layout_executivo(fig_volume, 'Volume mensal da população', titulo_eixo_x='Safra',
                         titulo_eixo_y='Quantidade de registros', mostrar_legenda=False)
fig_volume.show()
salvar_grafico(fig_volume, '01_01_volume_por_safra', PASTA_FIGURAS, salvar_png=False)

fig_evento = go.Figure(go.Scatter(
    x=tabela_safras['safra'], y=tabela_safras['taxa_evento'], mode='lines+markers',
    line=dict(color=CORES['principal'], width=3), marker=dict(size=8),
    hovertemplate='Safra: %{x|%Y-%m}<br>Taxa: %{y:.2%}<extra></extra>',
))
aplicar_layout_executivo(fig_evento, 'Taxa do evento adverso por safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Taxa do evento', mostrar_legenda=False)
aplicar_eixo_percentual(fig_evento)
fig_evento.show()
salvar_grafico(fig_evento, '01_02_taxa_evento_por_safra', PASTA_FIGURAS, salvar_png=False)
variacao_jan20 = tabela_safras.iloc[-1]['taxa_evento'] - tabela_safras.iloc[-2]['taxa_evento']
display(Markdown(
    f"**Análise/Interpretação:** O volume cresce de **{tabela_safras.iloc[0].quantidade_registros:,.0f}** para "
    f"**{tabela_safras.iloc[-1].quantidade_registros:,.0f}** registros. A taxa aumenta gradualmente de "
    f"**{tabela_safras.iloc[0].taxa_evento:.2%}** para **{tabela_safras.iloc[-2].taxa_evento:.2%}** até 2019-12, "
    f"com pequenas oscilações, e apresenta pico de **{tabela_safras.iloc[-1].taxa_evento:.2%}** em 2020-01 "
    f"(**{variacao_jan20:+.2%}** p.p. ante a safra anterior). Foi observada mudança de comportamento da "
    "população/target em 2020-01; sua causa não pode ser determinada com a base anonimizada."
))

**Análise/Interpretação:** O volume cresce de **13,936** para **18,318** registros. A taxa aumenta gradualmente de **10.85%** para **13.11%** até 2019-12, com pequenas oscilações, e apresenta pico de **17.47%** em 2020-01 (**+4.37%** p.p. ante a safra anterior). Foi observada mudança de comportamento da população/target em 2020-01; sua causa não pode ser determinada com a base anonimizada.

## Resultados

### Os padrões de missing mudaram entre as safras?

In [4]:
variaveis_missing = base.columns[base.isna().any()].tolist()
variaveis_missing.remove('safra') if 'safra' in variaveis_missing else None
tabela_missing_safra = (base.groupby('safra')[variaveis_missing].agg(lambda s: s.isna().mean())
                        .reset_index().melt(id_vars='safra', var_name='variavel', value_name='percentual_missing'))
resumo_missing = (tabela_missing_safra.groupby('variavel')['percentual_missing']
                  .agg(missing_minimo='min', missing_maximo='max', amplitude='max').reset_index())
resumo_missing['amplitude'] = resumo_missing['missing_maximo'] - resumo_missing['missing_minimo']
display(resumo_missing.sort_values('amplitude', ascending=False).style.format({
    'missing_minimo': '{:.2%}', 'missing_maximo': '{:.2%}', 'amplitude': '{:.2%}'}))
matriz_missing = tabela_missing_safra.pivot(index='variavel', columns='safra', values='percentual_missing')
fig_missing = px.imshow(
    matriz_missing, aspect='auto', color_continuous_scale=[CORES['fundo'], CORES['destaque']],
    labels={'x': 'Safra', 'y': 'Variável', 'color': 'Missing'}, zmin=0, zmax=matriz_missing.max().max(),
)
aplicar_layout_executivo(fig_missing, 'Percentual de missing por variável e safra',
                         titulo_eixo_x='Safra', titulo_eixo_y='Variável', altura=500)
fig_missing.update_coloraxes(colorbar_tickformat='.1%')
fig_missing.show()
salvar_grafico(fig_missing, '01_03_missing_por_safra', PASTA_FIGURAS, salvar_png=False)
cat13 = tabela_missing_safra.query("variavel == 'cat_var13'").sort_values('safra')
display(Markdown(
    f"**Análise/Interpretação:** `cat_var13` apresenta mudança estrutural gradual: o missing cai de "
    f"**{cat13.iloc[0].percentual_missing:.2%}** para **{cat13.iloc[-1].percentual_missing:.2%}** "
    f"(amplitude de **{cat13.percentual_missing.max()-cat13.percentual_missing.min():.2%}**). "
    "As demais variáveis têm ausência baixa e comparativamente estável. Este é um achado descritivo: não demonstra "
    "leakage nem define remoção, imputação ou criação de categoria. O comportamento deverá ser considerado na preparação e seleção futuras."
))

,variavel,missing_minimo,missing_maximo,amplitude
0,cat_var13,23.21%,78.33%,55.12%
2,cat_var2,0.37%,0.77%,0.40%
4,var4,0.55%,0.93%,0.39%
5,var8,0.15%,0.36%,0.21%
6,var9,0.11%,0.28%,0.17%
3,var11,0.00%,0.01%,0.01%
1,cat_var15,0.00%,0.01%,0.01%


**Análise/Interpretação:** `cat_var13` apresenta mudança estrutural gradual: o missing cai de **78.33%** para **23.21%** (amplitude de **55.12%**). As demais variáveis têm ausência baixa e comparativamente estável. Este é um achado descritivo: não demonstra leakage nem define remoção, imputação ou criação de categoria. O comportamento deverá ser considerado na preparação e seleção futuras.

### Os códigos especiais de `var12` têm comportamento temporal consistente?

**Comentário Técnico:** a comparação é descritiva. Associação com o alvo não determina leakage e os códigos permanecem intactos.

In [5]:
codigos_especiais = [99997, 99998, 99999]
grupo_var12 = pd.Series('demais', index=base.index, dtype='string')
mascara_especial_var12 = base['var12'].isin(codigos_especiais)
grupo_var12.loc[mascara_especial_var12] = base.loc[mascara_especial_var12, 'var12'].map(lambda valor: str(int(valor)))
tabela_var12 = (base.assign(grupo_var12=grupo_var12).groupby(['safra', 'grupo_var12'])[ALVO]
                .agg(quantidade='size', quantidade_eventos='sum', taxa_evento='mean').reset_index())
tabela_var12['percentual_safra'] = tabela_var12['quantidade'] / tabela_var12.groupby('safra')['quantidade'].transform('sum')
tabela_var12_total = (base.assign(grupo_var12=grupo_var12).groupby('grupo_var12')[ALVO]
                      .agg(quantidade='size', quantidade_eventos='sum', taxa_evento='mean').reset_index())
tabela_var12_total['percentual_total'] = tabela_var12_total['quantidade'] / len(base)
display(tabela_var12_total.style.format({'taxa_evento': '{:.2%}', 'percentual_total': '{:.2%}'}))
display(tabela_var12.style.format({'safra': lambda x: x.strftime('%Y-%m'), 'taxa_evento': '{:.2%}',
                                    'percentual_safra': '{:.2%}'}))
fig_var12_freq = px.line(tabela_var12, x='safra', y='percentual_safra', color='grupo_var12', markers=True,
                         labels={'grupo_var12': 'Código/grupo'})
aplicar_layout_executivo(fig_var12_freq, 'Participação dos códigos especiais de var12 por safra',
                         titulo_eixo_x='Safra', titulo_eixo_y='Participação na safra')
aplicar_eixo_percentual(fig_var12_freq)
fig_var12_freq.show()
salvar_grafico(fig_var12_freq, '01_04_var12_codigos_frequencia', PASTA_FIGURAS, salvar_png=False)
fig_var12_taxa = px.line(tabela_var12, x='safra', y='taxa_evento', color='grupo_var12', markers=True,
                         labels={'grupo_var12': 'Código/grupo'})
aplicar_layout_executivo(fig_var12_taxa, 'Taxa do evento por código especial de var12',
                         titulo_eixo_x='Safra', titulo_eixo_y='Taxa do evento')
aplicar_eixo_percentual(fig_var12_taxa)
fig_var12_taxa.show()
salvar_grafico(fig_var12_taxa, '01_05_var12_codigos_taxa_evento', PASTA_FIGURAS, salvar_png=False)
var99997 = tabela_var12.query("grupo_var12 == '99997'").sort_values('safra')
display(Markdown(
    f"**Análise/Interpretação:** A associação de `var12=99997` com maior risco aparece em todas as safras: sua taxa "
    f"varia de **{var99997.taxa_evento.min():.2%}** a **{var99997.taxa_evento.max():.2%}** e alcança "
    f"**{var99997.iloc[-1].taxa_evento:.2%}** em 2020-01. A elevação recente também ocorre no grupo `demais`. "
    "Trata-se de associação, não de leakage demonstrado. O significado e a disponibilidade temporal dos códigos seguem desconhecidos."
))

,grupo_var12,quantidade,quantidade_eventos,taxa_evento,percentual_total
0,99997,22724,5604,24.66%,11.36%
1,99998,58776,5446,9.27%,29.38%
2,99999,105177,10630,10.11%,52.58%
3,demais,13366,3538,26.47%,6.68%


,safra,grupo_var12,quantidade,quantidade_eventos,taxa_evento,percentual_safra
0,2019-01,99997,1697,370,21.80%,12.18%
1,2019-01,99998,2548,204,8.01%,18.28%
2,2019-01,99999,9219,825,8.95%,66.15%
3,2019-01,demais,472,113,23.94%,3.39%
4,2019-02,99997,1532,352,22.98%,11.07%
5,2019-02,99998,3465,275,7.94%,25.04%
6,2019-02,99999,8143,786,9.65%,58.85%
7,2019-02,demais,698,188,26.93%,5.04%
8,2019-03,99997,1596,350,21.93%,11.58%
9,2019-03,99998,3658,318,8.69%,26.54%


**Análise/Interpretação:** A associação de `var12=99997` com maior risco aparece em todas as safras: sua taxa varia de **21.63%** a **32.47%** e alcança **32.47%** em 2020-01. A elevação recente também ocorre no grupo `demais`. Trata-se de associação, não de leakage demonstrado. O significado e a disponibilidade temporal dos códigos seguem desconhecidos.

### Existem evidências de mudança na distribuição das features?

O PSI compara participações em bins entre uma população de referência e uma população de comparação. Aqui, a referência comum aos dois cenários é 2019-01 a 2019-08 e cada safra de 2019-09 a 2020-01 é comparada separadamente. Para numéricas, os limites são decis fixados exclusivamente pela referência; valores fora da faixa ficam nos bins extremos. Missing é um bin explícito e, em `var12`, cada código especial também é um bin separado. Para categóricas, usa-se a união das categorias observadas, preservando missing; categorias ausentes em um lado recebem `epsilon=1e-6`. O cálculo é `Σ(q-p)×ln(q/p)`. Bins nunca são recalculados pela safra comparada. Os resultados são diagnóstico temporal exploratório, sem threshold absoluto e sem exclusão de features.

In [6]:
def calcular_psi_continuo(referencia, comparacao, especiais=(), epsilon=1e-6):
    """Calcula PSI com quantis da referência e bins separados para missing/especiais."""
    ref = pd.Series(referencia)
    comp = pd.Series(comparacao)
    especiais = set(especiais)
    ref_regular = ref[ref.notna() & ~ref.isin(especiais)]
    limites = np.unique(ref_regular.quantile(np.linspace(0, 1, 11)).to_numpy())
    if len(limites) < 2:
        return np.nan
    limites[0], limites[-1] = -np.inf, np.inf
    def rotular(serie):
        resultado = pd.cut(serie, bins=limites, include_lowest=True).astype('string').fillna('__MISSING__')
        for valor in especiais:
            resultado.loc[serie.eq(valor)] = f'__ESPECIAL_{valor}__'
        return resultado
    categorias = rotular(ref).value_counts(normalize=True).index.union(rotular(comp).value_counts(normalize=True).index)
    p = rotular(ref).value_counts(normalize=True).reindex(categorias, fill_value=0).clip(lower=epsilon)
    q = rotular(comp).value_counts(normalize=True).reindex(categorias, fill_value=0).clip(lower=epsilon)
    return float(((q - p) * np.log(q / p)).sum())

def calcular_psi_categorico(referencia, comparacao, epsilon=1e-6):
    """Calcula PSI categórico preservando missing como categoria explícita."""
    ref = pd.Series(referencia).astype('string').fillna('__MISSING__')
    comp = pd.Series(comparacao).astype('string').fillna('__MISSING__')
    categorias = ref.unique().tolist() + [c for c in comp.unique() if c not in set(ref.unique())]
    p = ref.value_counts(normalize=True).reindex(categorias, fill_value=0).clip(lower=epsilon)
    q = comp.value_counts(normalize=True).reindex(categorias, fill_value=0).clip(lower=epsilon)
    return float(((q - p) * np.log(q / p)).sum())

In [7]:
fim_referencia_psi = pd.Timestamp('2019-08-01')
referencia = base[base['safra'] <= fim_referencia_psi]
safras_comparacao = [s for s in safras if s > fim_referencia_psi]
registros_psi = []
for variavel in VARIAVEIS_MODELO:
    for safra in safras_comparacao:
        comparacao = base[base['safra'].eq(safra)]
        if variavel in VARIAVEIS_NUMERICAS:
            especiais = codigos_especiais if variavel == 'var12' else ()
            psi = calcular_psi_continuo(referencia[variavel], comparacao[variavel], especiais)
            tipo = 'numerica'
        else:
            psi = calcular_psi_categorico(referencia[variavel], comparacao[variavel])
            tipo = 'categorica'
        registros_psi.append({'variavel': variavel, 'tipo': tipo, 'safra_comparacao': safra, 'psi': psi})
tabela_psi = pd.DataFrame(registros_psi)
resumo_psi = (tabela_psi.groupby(['variavel', 'tipo'])['psi'].agg(psi_maximo='max', psi_medio='mean')
              .reset_index().sort_values('psi_maximo', ascending=False))
display(resumo_psi.style.format({'psi_maximo': '{:.3f}', 'psi_medio': '{:.3f}'}))
matriz_psi = tabela_psi.pivot(index='variavel', columns='safra_comparacao', values='psi')
fig_psi = px.imshow(matriz_psi, aspect='auto', color_continuous_scale=[CORES['fundo'], CORES['destaque']],
                    labels={'x': 'Safra comparada', 'y': 'Variável', 'color': 'PSI'})
aplicar_layout_executivo(fig_psi, 'PSI exploratório versus referência 2019-01 a 2019-08',
                         titulo_eixo_x='Safra comparada', titulo_eixo_y='Feature', altura=650)
fig_psi.show()
salvar_grafico(fig_psi, '01_06_psi_features', PASTA_FIGURAS, salvar_png=False)
top_psi = resumo_psi.iloc[0]
display(Markdown(
    f"**Análise/Interpretação:** A maior mudança pelo PSI exploratório é de `{top_psi.variavel}` "
    f"(**PSI máximo {top_psi.psi_maximo:.3f}**). A ordenação deve ser lida junto das estatísticas mensais: "
    "PSI alto pode refletir mudança real, missing, códigos especiais ou categorias novas. Não implica exclusão da feature."
))

,variavel,tipo,psi_maximo,psi_medio
1,cat_var13,categorica,0.411,0.344
0,cat_var10,categorica,0.277,0.214
4,cat_var6,categorica,0.214,0.159
9,var3,numerica,0.073,0.050
12,var7,numerica,0.065,0.040
5,var1,numerica,0.040,0.032
7,var12,numerica,0.037,0.022
8,var14,numerica,0.035,0.016
2,cat_var15,categorica,0.030,0.024
3,cat_var2,categorica,0.012,0.004


**Análise/Interpretação:** A maior mudança pelo PSI exploratório é de `cat_var13` (**PSI máximo 0.411**). A ordenação deve ser lida junto das estatísticas mensais: PSI alto pode refletir mudança real, missing, códigos especiais ou categorias novas. Não implica exclusão da feature.

In [8]:
estatisticas_numericas = (base.groupby('safra')[VARIAVEIS_NUMERICAS]
    .agg(['count', 'mean', 'median', 'std', 'min', lambda s: s.quantile(.25), lambda s: s.quantile(.75), 'max']))
estatisticas_numericas.columns = ['__'.join([a, {'<lambda_0>':'p25','<lambda_1>':'p75'}.get(b,b)])
                                   for a,b in estatisticas_numericas.columns]
estatisticas_numericas = estatisticas_numericas.reset_index()
registros_categorias = []
for variavel in VARIAVEIS_CATEGORICAS:
    for safra, grupo in base.groupby('safra'):
        frequencias = grupo[variavel].astype('string').fillna('__MISSING__').value_counts(normalize=True)
        categorias_ref = set(referencia[variavel].astype('string').fillna('__MISSING__').unique())
        categorias_atual = set(frequencias.index)
        registros_categorias.append({
            'variavel': variavel, 'safra': safra, 'cardinalidade': len(categorias_atual),
            'categoria_dominante': frequencias.index[0], 'participacao_dominante': frequencias.iloc[0],
            'categorias_raras_menor_1pct': int((frequencias < .01).sum()),
            'categorias_novas_vs_referencia': len(categorias_atual - categorias_ref),
            'categorias_ausentes_vs_referencia': len(categorias_ref - categorias_atual),
        })
tabela_estabilidade_categoricas = pd.DataFrame(registros_categorias)
display(estatisticas_numericas.head())
display(tabela_estabilidade_categoricas.query("variavel == 'cat_var10'").style.format({'participacao_dominante': '{:.2%}'}))
cat10 = tabela_estabilidade_categoricas.query("variavel == 'cat_var10'").sort_values('safra')
fig_cat10 = px.line(cat10, x='safra', y='participacao_dominante', markers=True)
fig_cat10.update_traces(line_color=CORES['principal'])
aplicar_layout_executivo(fig_cat10, 'Participação da categoria dominante de cat_var10',
                         titulo_eixo_x='Safra', titulo_eixo_y='Participação dominante', mostrar_legenda=False)
aplicar_eixo_percentual(fig_cat10)
fig_cat10.show()
salvar_grafico(fig_cat10, '01_07_cat_var10_concentracao', PASTA_FIGURAS, salvar_png=False)
display(Markdown(
    f"**Análise/Interpretação:** A categoria dominante de `cat_var10` cai de "
    f"**{cat10.iloc[0].participacao_dominante:.2%}** para **{cat10.iloc[-1].participacao_dominante:.2%}**. "
    "Há diversificação gradual da distribuição, e não desaparecimento abrupto da variável. As estatísticas numéricas "
    "e a tabela categórica exportada permitem revisar todas as features sem produzir dezenas de gráficos."
))

,safra,var1__count,var1__mean,var1__median,var1__std,var1__min,var1__p25,var1__p75,var1__max,var3__count,var3__mean,var3__median,var3__std,var3__min,var3__p25,var3__p75,var3__max,var4__count,var4__mean,var4__median,var4__std,var4__min,var4__p25,var4__p75,var4__max,var5__count,var5__mean,var5__median,var5__std,var5__min,...,var9__median,var9__std,var9__min,var9__p25,var9__p75,var9__max,var11__count,var11__mean,var11__median,var11__std,var11__min,var11__p25,var11__p75,var11__max,var12__count,var12__mean,var12__median,var12__std,var12__min,var12__p25,var12__p75,var12__max,var14__count,var14__mean,var14__median,var14__std,var14__min,var14__p25,var14__p75,var14__max
0,2019-01-01,13936,383.850069,297.205,355.490752,0.0,140.5400,525.7375,5955.73,13936,441.459493,344.15,380.860422,5.00,176.1450,593.530,6443.41,13813,0.928485,1.0,0.198199,0.019753,1.0,1.0,3.325102,13936,0.857954,0.916667,0.178896,0.0,...,1.0,0.192375,0.0,0.833333,1.0,1.0,13936,0.794677,1.0,0.298725,0.010259,0.537631,1.0,1.079914,13936,96611.729131,99999.0,18089.433831,0.333333,99998.0,99999.0,99999.0,13936,0.810096,0.916667,0.204571,0.0,0.666667,1.0,1.0
1,2019-02-01,13838,403.342023,310.000,362.688916,0.0,149.0525,554.0575,4945.91,13838,456.143940,360.68,386.923598,5.25,181.2450,618.850,4945.91,13717,0.921418,1.0,0.249405,0.027894,1.0,1.0,11.512491,13838,0.864390,0.916667,0.172570,0.0,...,1.0,0.195800,0.0,0.833333,1.0,1.0,13838,0.802502,1.0,0.295601,0.000017,0.572628,1.0,1.000823,13838,94954.550911,99999.0,21885.507772,0.333333,99998.0,99999.0,99999.0,13838,0.810955,0.916667,0.205682,0.0,0.666667,1.0,1.0
2,2019-03-01,13783,403.458500,312.270,367.778645,0.0,150.3550,552.1600,7203.98,13783,457.918270,359.23,391.528182,5.25,184.5300,620.205,7761.27,13664,0.913795,1.0,0.213351,0.064535,1.0,1.0,2.000000,13783,0.873975,0.916667,0.168447,0.0,...,1.0,0.195793,0.0,0.833333,1.0,1.0,13783,0.804986,1.0,0.293033,0.000017,0.577259,1.0,1.000680,13783,94491.832668,99999.0,22811.500601,0.333333,99998.0,99999.0,99999.0,13783,0.818309,0.916667,0.205976,0.0,0.750000,1.0,1.0
3,2019-04-01,13825,413.861901,315.100,370.811389,0.0,153.6200,572.7700,4251.78,13825,470.156699,372.37,390.181788,5.00,189.0600,645.970,4251.78,13708,0.918521,1.0,0.220204,0.060062,1.0,1.0,6.567988,13825,0.881027,0.916667,0.163708,0.0,...,1.0,0.197761,0.0,0.833333,1.0,1.0,13824,0.817729,1.0,0.285415,0.014038,0.629504,1.0,1.000614,13825,93835.886860,99999.0,24048.009632,0.333333,99998.0,99999.0,99999.0,13825,0.823466,0.916667,0.202951,0.0,0.750000,1.0,1.0
4,2019-05-01,14182,412.699713,319.755,366.340199,0.0,153.0900,572.3375,4173.42,14182,467.913009,374.20,384.532746,5.16,187.9125,643.955,4371.36,14061,0.914404,1.0,0.218979,0.098605,1.0,1.0,7.299793,14182,0.885026,0.916667,0.160376,0.0,...,1.0,0.198926,0.0,0.833333,1.0,1.0,14182,0.825990,1.0,0.280925,0.004627,0.669815,1.0,1.123848,14182,93497.423483,99999.0,24654.907064,0.333333,99998.0,99999.0,99999.0,14182,0.826934,0.916667,0.199491,0.0,0.750000,1.0,1.0


,variavel,safra,cardinalidade,categoria_dominante,participacao_dominante,categorias_raras_menor_1pct,categorias_novas_vs_referencia,categorias_ausentes_vs_referencia
26,cat_var10,2019-01-01 00:00:00,10,0,96.33%,8,0,0
27,cat_var10,2019-02-01 00:00:00,10,0,94.57%,7,0,0
28,cat_var10,2019-03-01 00:00:00,10,0,92.58%,6,0,0
29,cat_var10,2019-04-01 00:00:00,10,0,90.37%,5,0,0
30,cat_var10,2019-05-01 00:00:00,10,0,88.00%,4,0,0
31,cat_var10,2019-06-01 00:00:00,10,0,84.42%,3,0,0
32,cat_var10,2019-07-01 00:00:00,10,0,80.72%,2,0,0
33,cat_var10,2019-08-01 00:00:00,10,0,77.33%,1,0,0
34,cat_var10,2019-09-01 00:00:00,10,0,73.70%,0,0,0
35,cat_var10,2019-10-01 00:00:00,10,0,73.04%,0,0,0


**Análise/Interpretação:** A categoria dominante de `cat_var10` cai de **96.33%** para **71.33%**. Há diversificação gradual da distribuição, e não desaparecimento abrupto da variável. As estatísticas numéricas e a tabela categórica exportada permitem revisar todas as features sem produzir dezenas de gráficos.

### Alguma variável pode conter informação indisponível no momento da decisão?

A resposta exige definição e linhagem de negócio. A estabilidade estatística não prova disponibilidade temporal.

In [9]:
registros_leakage = [
    {'variavel': COLUNA_ID, 'risco_leakage': 'não aplicável como feature',
     'motivo': 'identificador; decisão D005 determina exclusão dos atributos preditivos', 'status': 'sem indício'},
    {'variavel': COLUNA_SAFRA, 'risco_leakage': 'controle temporal',
     'motivo': 'usada para ordenação e estabilidade; não como atributo preditivo', 'status': 'sem indício'},
    {'variavel': ALVO, 'risco_leakage': 'crítico se usada como feature',
     'motivo': 'outcome posterior; somente alvo', 'status': 'sem indício no desenho atual'},
]
for variavel in VARIAVEIS_MODELO:
    if variavel == 'var12':
        risco = 'indeterminado; revisão prioritária'
        motivo = 'códigos especiais associados ao alvo; a associação não demonstra leakage nem disponibilidade pós-decisão'
        status = 'revisar'
    else:
        risco = 'indeterminado'
        motivo = 'nome anonimizado; documentação não informa janela de cálculo nem data de disponibilidade'
        status = 'depende de definição de negócio'
    registros_leakage.append({'variavel': variavel, 'risco_leakage': risco, 'motivo': motivo, 'status': status})
tabela_leakage = pd.DataFrame(registros_leakage)
display(tabela_leakage)
display(Markdown(
    "**Análise/Interpretação:** Não foi encontrado leakage temporal demonstrável nos dados, mas também não é possível "
    "certificar as 15 features como pré-decisão sem dicionário, fórmula e timestamp de disponibilidade. `var12` requer "
    "prioridade na revisão. Essa limitação é bloqueadora para aceitar definitivamente o conjunto de features, não para propor o split."
))

,variavel,risco_leakage,motivo,status
0,id,não aplicável como feature,identificador; decisão D005 determina exclusão...,sem indício
1,data_ref_safra,controle temporal,usada para ordenação e estabilidade; não como ...,sem indício
2,Ever30Mob6,crítico se usada como feature,outcome posterior; somente alvo,sem indício no desenho atual
3,var1,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
4,var3,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
5,var4,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
6,var5,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
7,var7,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
8,var8,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio
9,var9,indeterminado,nome anonimizado; documentação não informa jan...,depende de definição de negócio


**Análise/Interpretação:** Não foi encontrado leakage temporal demonstrável nos dados, mas também não é possível certificar as 15 features como pré-decisão sem dicionário, fórmula e timestamp de disponibilidade. `var12` requer prioridade na revisão. Essa limitação é bloqueadora para aceitar definitivamente o conjunto de features, não para propor o split.

### Qual cenário de split oferece delineamento temporal mais defensável?

São comparados 8/2/3 e 9/2/2 usando somente tempo, volume, eventos e estabilidade. Nenhuma performance de modelo participa da avaliação.

In [10]:
cenarios = {
    'A — 8/2/3': {
        'Treino': ('2019-01-01', '2019-08-01'), 'Validação': ('2019-09-01', '2019-10-01'),
        'OOT': ('2019-11-01', '2020-01-01'),
    },
    'B — 9/2/2': {
        'Treino': ('2019-01-01', '2019-09-01'), 'Validação': ('2019-10-01', '2019-11-01'),
        'OOT': ('2019-12-01', '2020-01-01'),
    },
}
registros_splits = []
for cenario, janelas in cenarios.items():
    for amostra, (inicio, fim) in janelas.items():
        recorte = base[base['safra'].between(inicio, fim)]
        registros_splits.append({
            'cenario': cenario, 'amostra': amostra, 'primeira_safra': recorte['safra'].min(),
            'ultima_safra': recorte['safra'].max(), 'quantidade_safras': recorte['safra'].nunique(),
            'quantidade_registros': len(recorte), 'percentual_populacao': len(recorte) / len(base),
            'quantidade_eventos': int(recorte[ALVO].sum()), 'taxa_evento': recorte[ALVO].mean(),
        })
tabela_comparacao_splits = pd.DataFrame(registros_splits)
display(tabela_comparacao_splits.style.format({
    'primeira_safra': lambda x: x.strftime('%Y-%m'), 'ultima_safra': lambda x: x.strftime('%Y-%m'),
    'percentual_populacao': '{:.2%}', 'taxa_evento': '{:.2%}'}))
for cenario in cenarios:
    recorte = tabela_comparacao_splits.query('cenario == @cenario')
    assert recorte['quantidade_registros'].sum() == len(base)
    assert recorte['quantidade_eventos'].sum() == base[ALVO].sum()
avaliacao_splits = pd.DataFrame([
    {'criterio': 'Dados para treino', 'A — 8/2/3': '8 safras; menor volume, ainda amplo',
     'B — 9/2/2': '9 safras; maior volume e diversidade temporal'},
    {'criterio': 'Representatividade da validação', 'A — 8/2/3': '2019-09 a 2019-10; imediatamente pós-treino',
     'B — 9/2/2': '2019-10 a 2019-11; incorpora nível de evento mais recente'},
    {'criterio': 'Extensão do OOT', 'A — 8/2/3': '3 safras; 2019-11 a 2020-01',
     'B — 9/2/2': '2 safras; 2019-12 a 2020-01'},
    {'criterio': 'Medição de deterioração', 'A — 8/2/3': 'Mais pontos mensais fora do desenvolvimento',
     'B — 9/2/2': 'Menos pontos, com maior proximidade temporal'},
    {'criterio': 'Safra 2020-01', 'A — 8/2/3': 'Preservada no OOT', 'B — 9/2/2': 'Preservada no OOT'},
    {'criterio': 'Robustez fora do tempo', 'A — 8/2/3': 'Favorece avaliação temporal com três safras',
     'B — 9/2/2': 'Favorece desenvolvimento com treino maior'},
])
display(avaliacao_splits)
display(Markdown(
    "**Recomendação técnica:** preferir o **Cenário A — 8/2/3**, pois 8 safras ainda fornecem uma base "
    "de treino ampla, enquanto três safras de OOT permitem avaliar deterioração com mais extensão temporal e "
    "mantêm 2020-01 completamente fora do desenvolvimento. O Cenário B continua defensável se houver prioridade "
    "por maior volume de treino. **Decisão pendente de validação humana.** Futuras avaliações devem reportar "
    "métricas agregadas e por safra, especialmente 2020-01."
))

,cenario,amostra,primeira_safra,ultima_safra,quantidade_safras,quantidade_registros,percentual_populacao,quantidade_eventos,taxa_evento
0,A — 8/2/3,Treino,2019-01,2019-08,8,114324,57.15%,13338,11.67%
1,A — 8/2/3,Validação,2019-09,2019-10,2,32621,16.31%,4093,12.55%
2,A — 8/2/3,OOT,2019-11,2020-01,3,53098,26.54%,7787,14.67%
3,B — 9/2/2,Treino,2019-01,2019-09,9,130458,65.21%,15331,11.75%
4,B — 9/2/2,Validação,2019-10,2019-11,2,33831,16.91%,4401,13.01%
5,B — 9/2/2,OOT,2019-12,2020-01,2,35754,17.87%,5486,15.34%


,criterio,A — 8/2/3,B — 9/2/2
0,Dados para treino,"8 safras; menor volume, ainda amplo",9 safras; maior volume e diversidade temporal
1,Representatividade da validação,2019-09 a 2019-10; imediatamente pós-treino,2019-10 a 2019-11; incorpora nível de evento m...
2,Extensão do OOT,3 safras; 2019-11 a 2020-01,2 safras; 2019-12 a 2020-01
3,Medição de deterioração,Mais pontos mensais fora do desenvolvimento,"Menos pontos, com maior proximidade temporal"
4,Safra 2020-01,Preservada no OOT,Preservada no OOT
5,Robustez fora do tempo,Favorece avaliação temporal com três safras,Favorece desenvolvimento com treino maior


**Recomendação técnica:** preferir o **Cenário A — 8/2/3**, pois 8 safras ainda fornecem uma base de treino ampla, enquanto três safras de OOT permitem avaliar deterioração com mais extensão temporal e mantêm 2020-01 completamente fora do desenvolvimento. O Cenário B continua defensável se houver prioridade por maior volume de treino. **Decisão pendente de validação humana.** Futuras avaliações devem reportar métricas agregadas e por safra, especialmente 2020-01.

## Takeaways e exportação

**Fato observado:** há crescimento de volume, aumento gradual da taxa do evento e um salto em 2020-01; `cat_var13` e `cat_var10` mudam substancialmente ao longo do tempo.

**Interpretação:** a base contém drift observado suficiente para justificar validação temporal rigorosa. Isso não determina exclusão de variáveis nem tratamento.

**Hipótese:** explicações operacionais, de captura, política ou composição exigiriam informação externa. Nenhuma delas é afirmada com a base anonimizada.

**Recomendação técnica, pendente de validação humana:** preferir 8/2/3 pela extensão do OOT, mantendo 9/2/2 como alternativa defensável. Após aprovação, congelar o OOT e impedir seu uso em seleção, tratamentos orientados pelo alvo, tuning, hiperparâmetros e escolha do champion. Antes do baseline, confirmar disponibilidade temporal das features e significado dos códigos de `var12`.

In [11]:
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
tabelas_exportar = {
    '01_event_rate_by_vintage.csv': tabela_safras,
    '01_missing_by_vintage.csv': tabela_missing_safra,
    '01_var12_special_codes_total.csv': tabela_var12_total,
    '01_var12_special_codes_by_vintage.csv': tabela_var12,
    '01_feature_stability_psi.csv': tabela_psi,
    '01_feature_stability_psi_summary.csv': resumo_psi,
    '01_numeric_statistics_by_vintage.csv': estatisticas_numericas,
    '01_categorical_stability_by_vintage.csv': tabela_estabilidade_categoricas,
    '01_leakage_review.csv': tabela_leakage,
    '01_split_summary.csv': tabela_comparacao_splits,
    '01_split_qualitative_comparison.csv': avaliacao_splits,
}
for nome, tabela in tabelas_exportar.items():
    tabela.to_csv(PASTA_TABELAS / nome, index=False, encoding='utf-8-sig')
assert base_original.equals(base.drop(columns=['safra'])), 'Colunas originais foram alteradas.'
display(Markdown(
    f"**Status da execução:** {len(tabelas_exportar)} tabelas `01_*` exportadas. "
    "Nenhum dado bruto, valor especial ou missing foi modificado. O notebook termina antes de preprocessing e modelagem."
))

**Status da execução:** 11 tabelas `01_*` exportadas. Nenhum dado bruto, valor especial ou missing foi modificado. O notebook termina antes de preprocessing e modelagem.